# silver_perdas 

- **Entrada:** `bronze.tb_geracao_distribuida`, `bronze.tb_inspecoes` e `silver.tb_unidades_consumidoras`.
- **Saída:** duas Silver, duas quarentenas (`tb_*_quarentena`) e registros em `silver.tb_log_qualidade`.
- **Dependência:** executar após `silver_cadastro`. Importa `%run ./funcoes_comuns` na primeira célula.
- **Regra de segurança:** **preparar e validar AMBAS as fontes antes de gravar QUALQUER tabela**. Mesmo assim, duas gravações separadas não são uma transação atômica conjunta.
- **Regras importantes:** valores atípicos e datas fora da aba são **sinalizados, nunca inventados/corrigidos sem evidência**. Nenhum dado excluído sem contagem/reconciliação.
- **Privacidade:** não subir dados, outputs nem identificadores de clientes ao GitHub público.


In [0]:
%run ./funcoes_comuns

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text('catalog', 'workspace')
CATALOGO = dbutils.widgets.get('catalog')
NOTEBOOK = 'silver_perdas'
UC = spark.table(f'{CATALOGO}.silver.tb_unidades_consumidoras')

MAPA_RESULTADO = {'CONFIRMADA':'IRREGULARIDADE CONFIRMADA','NADA CONSTA':'SEM IRREGULARIDADE'}
RESULTADOS_VALIDOS = ['IRREGULARIDADE CONFIRMADA', 'SEM IRREGULARIDADE']
MODALIDADES_VALIDAS = ['GERACAO NA PROPRIA UC', 'AUTOCONSUMO REMOTO', 'GERACAO COMPARTILHADA']
DATA_LIMITE_HISTORICO = '2026-06-30'  # apenas flag informativa; não excluir

# Critério exploratório deste conjunto, NÃO uma regra setorial/regulatória.
# Há um intervalo vazio entre 75 e 3.049,99 kWp no arquivo original;
# valores > 1.000 com injeção mensal < 1 kWh/kWp parecem erro de escala x1000.
LIMIAR_POTENCIA_SUSPEITA_KWP = 1000
LIMIAR_RELACAO_INJECAO_POTENCIA = 1

def contar_alterados(df, original, expressao_nova):
    return df.filter(~F.col(original).eqNullSafe(expressao_nova)).count()

def contar_data_nao_iso(df, nome):
    return df.filter(F.col(nome).isNotNull() & ~F.col(nome).rlike(r'^\d{4}-\d{2}-\d{2}$')).count()


## Parte A — Geração distribuída
Ler, tratar tipos e domínio, deduplicar, sinalizar valores suspeitos, checar integridade e gravar.

In [0]:
# === GERAÇÃO DISTRIBUÍDA: preparar e validar; NÃO GRAVAR AINDA ===
gd = ler_bronze(CATALOGO,'geracao_distribuida')
N_GD = gd.count()
REMOVIDAS_GD = {}
COLS_GD = [c for c in gd.columns if not c.startswith('_')]
COLS_GD_OBRIGATORIAS = ['id_uc','potencia_kwp','data_conexao','modalidade','energia_injetada_kwh_media_mensal']
assert set(COLS_GD_OBRIGATORIAS).issubset(COLS_GD),'Colunas de GD não coincidem com a Bronze esperada'
assert {'_source_file','_ingestion_timestamp'}.issubset(gd.columns),'Rastreio Bronze ausente'

# Passo 1: cabeçalho e espaços em branco
antes = gd.count()
gd = remover_cabecalho_repetido(gd)
REMOVIDAS_GD['cabecalho'] = antes - gd.count()
registrar('geracao_distribuida','cabeçalho repetido','remover_cabecalho_repetido','REMOVIDO',REMOVIDAS_GD['cabecalho'])
gd = aparar_textos(gd)

# Passo 2: tipos. Não usar cast diretamente sobre texto bruto.
# Medir a grafia com vírgula antes de converter.
n_pot_virgula = gd.filter(F.col('potencia_kwp').contains(',')).count()
n_energia_virgula = gd.filter(F.col('energia_injetada_kwh_media_mensal').contains(',')).count()
n_data_nao_iso = contar_data_nao_iso(gd,'data_conexao')
for col in ('potencia_kwp','energia_injetada_kwh_media_mensal'):
    gd = gd.withColumn(col,valores_para_nulo(col))
checar_parse(gd,['potencia_kwp','energia_injetada_kwh_media_mensal'],parse_decimal,'geracao_distribuida')
checar_parse(gd,['data_conexao'],parse_data,'geracao_distribuida')
gd = gd.withColumns({
    'potencia_kwp':parse_decimal('potencia_kwp'),
    'energia_injetada_kwh_media_mensal':parse_decimal('energia_injetada_kwh_media_mensal'),
    'data_conexao':parse_data('data_conexao')
})
registrar('geracao_distribuida','potencia com vírgula decimal','parse_decimal','CORRIGIDO',n_pot_virgula)
registrar('geracao_distribuida','energia injetada com vírgula decimal','parse_decimal','CORRIGIDO',n_energia_virgula)
registrar('geracao_distribuida','data de conexão não ISO','parse_data','CORRIGIDO',n_data_nao_iso)

# Passo 3: padronização de domínio e chaves
nova_modalidade = mapear_dominio('modalidade',{})
n_modalidade = contar_alterados(gd,'modalidade',nova_modalidade)
gd = gd.withColumn('modalidade',nova_modalidade).withColumn('id_uc',padronizar_texto('id_uc'))
registrar('geracao_distribuida','modalidade em grafias diferentes','mapear_dominio','CORRIGIDO',n_modalidade)

# Passo 4: duplicatas exatas de negócio, sem considerar colunas de rastreio
antes = gd.count()
gd = deduplicar(gd)
REMOVIDAS_GD['duplicatas'] = antes - gd.count()
registrar('geracao_distribuida','duplicatas exatas','deduplicar(df)','REMOVIDO',REMOVIDAS_GD['duplicatas'])

# Passo 5: flags. Potência x1000 é HIPÓTESE; não dividir e não remover.
pot = F.col('potencia_kwp')
inj = F.col('energia_injetada_kwh_media_mensal')
cond_escala_suspeita = ((pot > F.lit(LIMIAR_POTENCIA_SUSPEITA_KWP)) &
                        (F.when(pot > 0, inj / pot).otherwise(F.lit(None)) < F.lit(LIMIAR_RELACAO_INJECAO_POTENCIA)))
gd = criar_flag(gd,'flag_potencia_escala_suspeita',cond_escala_suspeita)
gd = criar_flag(gd,'flag_potencia_nao_positiva',pot <= 0)
gd = criar_flag(gd,'flag_energia_injetada_negativa',inj < 0)
gd = criar_flag(gd,'flag_data_conexao_fora_historico',F.col('data_conexao')>F.lit(DATA_LIMITE_HISTORICO))
FLAGS_GD = {
 'flag_potencia_escala_suspeita':('potência incompatível com a energia mensal (possível escala x1000)',
                                   'potencia_kwp > 1000 AND energia_injetada_kwh_media_mensal/potencia_kwp < 1; manter valor original'),
 'flag_potencia_nao_positiva':('potência não positiva','potencia_kwp <= 0'),
 'flag_energia_injetada_negativa':('energia injetada negativa','energia_injetada_kwh_media_mensal < 0'),
 'flag_data_conexao_fora_historico':('conexão após fim do histórico','data_conexao > 2026-06-30')
}
registrar_flags('geracao_distribuida',gd,FLAGS_GD)

# Passo 6: órfãos
# Manter separadas as contagens. Não descartar silenciosamente.
gd,gd_q = separar_orfaos(gd,UC,'id_uc',motivo='id_uc inexistente em silver.tb_unidades_consumidoras')
registrar('geracao_distribuida','UC inexistente no cadastro','separar_orfaos por id_uc','QUARENTENA',gd_q.count())

# Passo 7: garantir contrato e conciliação antes de qualquer gravação
checar_unicidade(gd,['id_uc'],'geracao_distribuida')
checar_sem_nulos(gd,
                COLS_GD_OBRIGATORIAS + list(FLAGS_GD)
                + ['_source_file', '_ingestion_timestamp'],
                'geracao_distribuida')
checar_dominio(gd,'modalidade',MODALIDADES_VALIDAS,'geracao_distribuida')
reconciliar('geracao_distribuida',N_GD,gd.count(),gd_q.count(),REMOVIDAS_GD)
gd_final = gd.select(*COLS_GD,*FLAGS_GD,'_source_file','_ingestion_timestamp')
print('GD VALIDADA:',gd_final.count(),'linhas; quarentena:',gd_q.count())


## Parte B — Inspeções
A Bronze já une as três abas; `source_sheet` é renomeada para `_source_sheet` na Silver, preservando a coluna de rastreio e deve ser preservada.

**Atenção:** o arquivo contém 1.389 linhas (não 1.392). Não usar a aba do Excel como referência temporal de validade.

In [0]:
# === INSPEÇÕES: preparar e validar; NÃO GRAVAR AINDA ===
ins = ler_bronze(CATALOGO,'inspecoes')
# A Bronze atual sanitiza nomes de colunas: _source_sheet vira source_sheet.
# Restaurar o metadado no dataframe da Silver sem modificar a Bronze nem inventar seu valor.
if '_source_sheet' not in ins.columns and 'source_sheet' in ins.columns:
    ins = ins.withColumnRenamed('source_sheet', '_source_sheet')
faltantes_rastreio = {'_source_sheet', '_source_file', '_ingestion_timestamp'} - set(ins.columns)
if faltantes_rastreio:
    raise ValueError(
        f'[inspecoes] Rastreio Bronze incompleto: {sorted(faltantes_rastreio)}. '
        f'Colunas encontradas: {ins.columns}. Verifique a task de ingestão do Excel.'
    )
N_INS = ins.count()
REMOVIDAS_INS = {}
COLS_INS = [c for c in ins.columns if not c.startswith('_')]
COLS_INS_OBRIGATORIAS = ['id_inspecao','id_uc','data','inspetor','resultado','kwh_recuperado']
assert set(COLS_INS_OBRIGATORIAS+['tipo_irregularidade']).issubset(COLS_INS),'Colunas INS faltando'

# Passo 1: estrutura
antes = ins.count()
ins = remover_cabecalho_repetido(ins)
REMOVIDAS_INS['cabecalho'] = antes-ins.count()
registrar('inspecoes','cabeçalho repetido','remover_cabecalho_repetido','REMOVIDO',REMOVIDAS_INS['cabecalho'])
# A exploração encontrou espaços nas extremidades de `inspetor`; registrar o tratamento.
n_espacos_inspetor = contar_espacos_nas_pontas(ins, ['inspetor'])
ins = aparar_textos(ins)
registrar('inspecoes', 'espaços nas pontas em inspetor',
          'aparar_textos', 'CORRIGIDO', n_espacos_inspetor)

# Passo 2: formatos mistos
n_virgula = ins.filter(F.col('kwh_recuperado').contains(',')).count()
n_data_nao_iso = contar_data_nao_iso(ins,'data')
ins = ins.withColumn('kwh_recuperado',valores_para_nulo('kwh_recuperado'))
checar_parse(ins,['kwh_recuperado'],parse_decimal,'inspecoes')
checar_parse(ins,['data'],parse_data,'inspecoes')
ins = ins.withColumns({'kwh_recuperado':parse_decimal('kwh_recuperado'),'data':parse_data('data')})
registrar('inspecoes','kwh recuperado com vírgula decimal','parse_decimal','CORRIGIDO',n_virgula)
registrar('inspecoes','data em formato não ISO','parse_data','CORRIGIDO',n_data_nao_iso)

# Passo 3: normalizar resultados e marcadores de ausência
novo_res = mapear_dominio('resultado',{'CONFIRMADA':'IRREGULARIDADE CONFIRMADA','NADA CONSTA':'SEM IRREGULARIDADE'})
n_res = contar_alterados(ins,'resultado',novo_res)
ins = ins.withColumn('resultado',novo_res)
registrar('inspecoes','resultado com sinônimo ou caixa diferente','mapear_dominio','CORRIGIDO',n_res)

novo_tipo = valores_para_nulo('tipo_irregularidade')
n_tipo_nulo = contar_alterados(ins,'tipo_irregularidade',novo_tipo)
ins = ins.withColumn('tipo_irregularidade',novo_tipo)
ins = ins.withColumn('tipo_irregularidade',padronizar_texto('tipo_irregularidade'))
registrar('inspecoes','marcador de ausência em tipo_irregularidade','valores_para_nulo + padronizar_texto','CORRIGIDO',n_tipo_nulo)

# O campo inspetor recebe limpeza textual; o identificador da UC é tratado sem inventar valores.
ins = ins.withColumns({c:padronizar_texto(c) for c in ['id_inspecao','id_uc','inspetor']})

# Passo 4: duplicatas exatas (rastreio da aba não compõe a chave)
antes = ins.count()
ins = deduplicar(ins)
REMOVIDAS_INS['duplicatas'] = antes-ins.count()
registrar('inspecoes','duplicatas exatas','deduplicar(df)','REMOVIDO',REMOVIDAS_INS['duplicatas'])

# Passo 5: indicadores de inconsistência - preservar resultado e energia originais
ins = criar_flag(ins,'flag_recuperacao_negativa',F.col('kwh_recuperado') < 0)
ins = criar_flag(ins,'flag_resultado_tipo_inconsistente',
    (F.col('resultado')=='SEM IRREGULARIDADE') & F.col('tipo_irregularidade').isNotNull())
ins = criar_flag(ins,'flag_confirmada_sem_tipo',
    (F.col('resultado')=='IRREGULARIDADE CONFIRMADA') & F.col('tipo_irregularidade').isNull())
ins = criar_flag(ins,'flag_recuperacao_sem_irregularidade',
    (F.col('resultado')=='SEM IRREGULARIDADE') & (F.col('kwh_recuperado')>0))
ins = criar_flag(ins,'flag_inspecao_fora_historico',F.col('data')>F.lit(DATA_LIMITE_HISTORICO))

# Divergência entre período nomeado na aba do Excel e a data da inspeção.
# A aba NÃO altera a validade do registro; marcação puramente informativa.
mes = F.month('data')
fora_aba = (
 ((F.col('_source_sheet')=='ago-set') & ~mes.isin(8,9)) |
 ((F.col('_source_sheet')=='out-nov') & ~mes.isin(10,11)) |
 ((F.col('_source_sheet')=='dez') & (mes != 12))
)
ins = criar_flag(ins,'flag_data_fora_periodo_aba',fora_aba)
FLAGS_INS = {
 'flag_recuperacao_negativa':('energia recuperada negativa','kwh_recuperado < 0'),
 'flag_resultado_tipo_inconsistente':('sem irregularidade mas tipo preenchido','resultado SEM IRREGULARIDADE e tipo não nulo'),
 'flag_confirmada_sem_tipo':('irregularidade confirmada sem tipo preenchido','resultado CONFIRMADA e tipo nulo'),
 'flag_recuperacao_sem_irregularidade':('energia recuperada sem irregularidade','resultado SEM IRREGULARIDADE e kwh_recuperado > 0'),
 'flag_inspecao_fora_historico':('data após fim do histórico','data > 2026-06-30'),
 'flag_data_fora_periodo_aba':('data não coincide com o período da aba Excel',
                              "ago-set fora ago/set, out-nov fora out/nov, dez fora dez; manter data real")
}
registrar_flags('inspecoes',ins,FLAGS_INS)

# Passo 6: separar órfãos
ins,ins_q = separar_orfaos(ins,UC,'id_uc',motivo='id_uc inexistente em silver.tb_unidades_consumidoras')
registrar('inspecoes','UC inexistente no cadastro','separar_orfaos por id_uc','QUARENTENA',ins_q.count())

# Passo 7: checar e reconciliar sem ter gravado nenhuma das duas tabelas ainda
checar_unicidade(ins,['id_inspecao'],'inspecoes')
checar_sem_nulos(ins,
                COLS_INS_OBRIGATORIAS + list(FLAGS_INS)
                + ['_source_sheet', '_source_file', '_ingestion_timestamp'],
                'inspecoes')
checar_dominio(ins,'resultado',RESULTADOS_VALIDOS,'inspecoes')
reconciliar('inspecoes',N_INS,ins.count(),ins_q.count(),REMOVIDAS_INS)
ins_final = ins.select(*COLS_INS,*FLAGS_INS,'_source_sheet','_source_file','_ingestion_timestamp')
print('INSPEÇÕES VALIDADAS:',ins_final.count(),'linhas; quarentena:',ins_q.count())


## Gravação e conferência final

As duas fontes já foram validadas e reconciliadas. Nesta etapa, gravamos as duas tabelas Silver e suas quarentenas, e por último registramos o log de qualidade. Este notebook não substitui um teste de execução no Databricks: é necessário **Run all** no seu workspace pessoal antes de compartilhar no GitHub.


In [0]:
# SOMENTE AGORA gravar: todas as validações e reconciliações foram concluídas.
print('Iniciando gravação das duas fontes já validadas...')
resultado_gd = gravar_silver(gd_final,CATALOGO,'geracao_distribuida',quarentena=gd_q)
resultado_ins = gravar_silver(ins_final,CATALOGO,'inspecoes',quarentena=ins_q)

# Checagens após gravação (detectam qualquer diferença entre o DF e Delta persistido)
assert resultado_gd['silver']==gd_final.count()
assert resultado_gd['quarentena']==gd_q.count()
assert resultado_ins['silver']==ins_final.count()
assert resultado_ins['quarentena']==ins_q.count()

gravar_log(CATALOGO,NOTEBOOK)

print('SAÍDAS:',resultado_gd,resultado_ins)
for nome in ('geracao_distribuida','inspecoes'):
    tab = f'{CATALOGO}.silver.tb_{nome}'
    tab_q = f'{CATALOGO}.silver.tb_{nome}_quarentena'
    print(tab,'linhas:',spark.table(tab).count(),'| quarentena:',spark.table(tab_q).count())
    spark.table(tab).printSchema()
display(spark.table(f'{CATALOGO}.silver.tb_log_qualidade')
             .filter(F.col('notebook')==NOTEBOOK).orderBy('tabela','destino'))


## Conferência de consistência para o time de negócio

- Na GD, `flag_potencia_escala_suspeita` **não altera `potencia_kwp`**. Quem desenvolver a Gold deve considerar o impacto dos 35 valores atípicos em indicadores de potência.
- Nas inspeções, `flag_data_fora_periodo_aba` não invalida a inspeção; as análises temporais devem usar a coluna `data`.
- Uma inspeção com resultado `SEM IRREGULARIDADE` pode ter legitimamente `kwh_recuperado = 0` e `tipo_irregularidade = null`.


In [0]:
print('Resumo agregado de qualidade (sem identificação de UCs):')
display(spark.table(f'{CATALOGO}.silver.tb_geracao_distribuida').groupBy('flag_potencia_escala_suspeita').count())
display(spark.table(f'{CATALOGO}.silver.tb_inspecoes').groupBy('resultado','flag_data_fora_periodo_aba').count())


## Indicadores de revisão para a Gold (agregados)

A soma de `potencia_kwp` **inclui** os registros suspeitos. Comparar os cenários antes de usar potência total ou média no dashboard. Estes números **não** são correções automáticas; a Silver preserva a fonte.

In [0]:
gd_tab = spark.table(f'{CATALOGO}.silver.tb_geracao_distribuida')
ins_tab = spark.table(f'{CATALOGO}.silver.tb_inspecoes')

# Consolida grandezas sem divulgar clientes, inspetores ou identificadores.
display(gd_tab.agg(
    F.count('*').alias('ucs_com_gd'),
    F.sum('potencia_kwp').alias('potencia_total_reportada_kwp'),
    F.sum(F.when(F.col('flag_potencia_escala_suspeita'), F.col('potencia_kwp'))
          .otherwise(F.lit(0))).alias('potencia_em_registros_suspeitos_kwp'),
    F.sum(F.when(~F.col('flag_potencia_escala_suspeita'), F.col('potencia_kwp'))
          .otherwise(F.lit(0))).alias('potencia_sem_suspeitos_kwp'),
    F.sum(F.col('flag_potencia_escala_suspeita').cast('long')).alias('ucs_suspeitas'),
))

display(ins_tab.groupBy('resultado').agg(
    F.count('*').alias('inspecoes'),
    F.sum('kwh_recuperado').alias('kwh_recuperado_total'),
    F.sum(F.col('flag_data_fora_periodo_aba').cast('long'))
      .alias('fora_do_periodo_da_aba'),
).orderBy('resultado'))
